# 23 · Human-in-the-Loop Approval

An autonomous agent that can act should not run **risky** tools unchecked. The
`approve_tool` hook is called before every tool executes; return `False` to
**veto** the call — the agent receives a denial as its observation and adapts,
instead of crashing.

This pairs naturally with Phase 4's tool **annotations** (`readOnlyHint` /
`destructiveHint`): auto-approve read-only tools, gate the rest.

> Runs on Ollama.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
root = next(x for x in [p, *p.parents] if (x / 'agents_lab').exists())
sys.path.insert(0, str(root))
import agents_lab  # noqa: F401  (bootstraps sibling study_buddy + rag_lab)

A simple policy: auto-approve the calculator; require (simulated) human
confirmation for anything else. Here the policy denies non-calculator tools.

In [ ]:
from agents_lab import Agent, default_tools

APPROVED = {'calculator'}  # in a real app, prompt a human here
def approve_tool(name, args):
    ok = name in APPROVED
    print(f'  approval: {name}({args}) -> {"ALLOW" if ok else "DENY"}')
    return ok

agent = Agent(provider='ollama', tools=default_tools(), approve_tool=approve_tool, max_steps=5)
result = agent.run('First compute 8*9 with the calculator, then tell me the current time.')
print('\nFINAL:', result.answer)

Notice the `current_time` call is **denied**, and the agent still produces a
sensible answer from what it was allowed to do.

To turn this into a real gate, replace the policy body with an actual prompt
(CLI `input()`, a UI confirm dialog, a Slack approval, ...).

**Takeaway:** an approval hook is the smallest, highest-leverage safety control
for an acting agent — a human stays on the loop for anything consequential.

Next → `capstone.ipynb`